In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from typing import Any

from langchain.tools import tool
from langchain_community.utilities import SQLDatabase
from sqlalchemy.exc import SQLAlchemyError
from tavily import TavilyClient

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)


@tool
def sql_query(query: str) -> Any:
    """Obtain information from the database using SQL queries"""
    try:
        return db.run(query)
    except SQLAlchemyError as e:
        return f"Error: {e}"

In [3]:
from dataclasses import dataclass


@dataclass
class UserRole:
    user_role: str = "external"

In [17]:
from _collections_abc import Callable

from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call


@wrap_model_call
def dynamic_tool_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role      # type: ignore
    
    if user_role == "internal":
        pass                      # internal users get access to all tools
    else:
        tool = [web_search]       # external users only get access to web search
        request = request.override(tools=tool)         # type: ignore

    return handler(request)

In [18]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],     # type: ignore
    context_schema=UserRole
)

In [19]:
from langchain.messages import HumanMessage

message = HumanMessage(content="How many artists are in the database?")

response = agent.invoke(
    {"messages": [message]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I don’t have access to your database, so I can’t see the exact number. Could you tell me which database you’re using and which table stores the artists? In the meantime, here are common queries you can run:

- SQL (MySQL, PostgreSQL, SQLite, etc.) to count total artists:
  SELECT COUNT(*) AS artist_count FROM artists;

- SQL to count distinct artists (by id):
  SELECT COUNT(DISTINCT artist_id) AS artist_count FROM artworks;

- If you want distinct by name:
  SELECT COUNT(DISTINCT name) AS artist_count FROM artists;

- If you’re using MongoDB:
  db.artists.countDocuments({})

If you share the table/collection name and what you want counted (total rows vs. unique artists, and any filters like active artists), I can tailor the exact query. If you’d like, you can paste the schema or a sample, and I’ll draft the precise query.


In [9]:
response = agent.invoke(
    {"messages": [message]},
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)

275 artists.

Note: The table is named Artist (not artists). Your initial query used a different (plural) name, which caused the error. If you want, I can show the first few artist records or run additional counts (e.g., by genre).


In [12]:
from pprint import pprint

pprint(response["messages"])

[HumanMessage(content='How many artists are in the database?', additional_kwargs={}, response_metadata={}, id='1e6fdd39-8fbf-4366-ac72-1daed8166d6b'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 415, 'prompt_tokens': 155, 'total_tokens': 570, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EAdOSTpgFHVG69RO06dRpCXSikMQI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fe205-23fc-7d13-af2f-63cb87e16411-0', tool_calls=[{'name': 'sql_query', 'args': {'query': 'SELECT COUNT(*) AS artist_count FROM artists;'}, 'id': 'call_yuO9OmEUNqgvAcTBVgGq6Nj7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'

In [13]:
from langchain.messages import ToolMessage

for message in response['messages']:
    if isinstance(message, ToolMessage):
        for field, value in message:
            print(f"{field}={value!r}")
        print()

content='Error: (sqlite3.OperationalError) no such table: artists\n[SQL: SELECT COUNT(*) AS artist_count FROM artists;]\n(Background on this error at: https://sqlalche.me/e/20/e3q8)'
additional_kwargs={}
response_metadata={}
type='tool'
name='sql_query'
id='ab979ed5-959d-4126-8e27-b57818ab6fd4'
tool_call_id='call_yuO9OmEUNqgvAcTBVgGq6Nj7'
artifact=None
status='success'

content="[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]"
additional_kwargs={}
response_metadata={}
type='tool'
name='sql_query'
id='61d0f267-ccf4-43a9-b881-cf708f962b08'
tool_call_id='call_DioPjFNFTcG4Vb4YxYVBeVbe'
artifact=None
status='success'

content='[(275,)]'
additional_kwargs={}
response_metadata={}
type='tool'
name='sql_query'
id='173845eb-3404-4c9d-a173-5ce9b0aaefdc'
tool_call_id='call_eWJWxf7Snq1JsHBwJtsRKSIg'
artifact=None
status='success'



In [16]:
for message in response['messages']:
    if message:
        print(type(message))
        for field, value in message:
            print(f"{field}={value!r}")
        print()

<class 'langchain_core.messages.human.HumanMessage'>
content='How many artists are in the database?'
additional_kwargs={}
response_metadata={}
type='human'
name=None
id='1e6fdd39-8fbf-4366-ac72-1daed8166d6b'

<class 'langchain_core.messages.ai.AIMessage'>
content=''
additional_kwargs={'refusal': None}
response_metadata={'token_usage': {'completion_tokens': 415, 'prompt_tokens': 155, 'total_tokens': 570, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EAdOSTpgFHVG69RO06dRpCXSikMQI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}
type='ai'
name=None
id='lc_run--019fe205-23fc-7d13-af2f-63cb87e16411-0'
tool_calls=[{'name': 'sql_query', 'args': {'query': 'SELECT COUNT(*) AS artist_count FROM arti